# NHANES - National Health and Nutrition Health Survey 2013-1014 Age Prediction Subset

## Preprocessing

https://archive.ics.uci.edu/dataset/887/national+health+and+nutrition+health+survey+2013-2014+(nhanes)+age+prediction+subset
* to assess health and nutritional status of adults and children in USA
* pre-processing performed: respondents 65 yrs old+ were labeled as senior, the ones under 65- as non-senior

In [1]:
import pandas as pd
df = pd.read_csv("NHANES_age_prediction.csv")

In [2]:
# check if the number of rows matches the number of unique SEQN values
n_rows = len(df)
n_unique = df["SEQN"].nunique(dropna=False)
# Explicit boolean check
is_unique = df["SEQN"].is_unique
print(f"SEQN is unique: {is_unique}")
# convert SEQN from float to int
df["SEQN"] = df["SEQN"].astype(int)
df = df.rename(columns={"SEQN": "id"})
#df = df.set_index("id", verify_integrity=True)
#df.head(3)

SEQN is unique: True


In [3]:
df.keys()

Index(['id', 'age_group', 'RIDAGEYR', 'RIAGENDR', 'PAQ605', 'BMXBMI', 'LBXGLU',
       'DIQ010', 'LBXGLT', 'LBXIN'],
      dtype='object')

In [4]:
df = df.drop(columns="id")

In [5]:
df.keys()

Index(['age_group', 'RIDAGEYR', 'RIAGENDR', 'PAQ605', 'BMXBMI', 'LBXGLU',
       'DIQ010', 'LBXGLT', 'LBXIN'],
      dtype='object')

In [6]:
df.head(10)

,age_group,RIDAGEYR,RIAGENDR,PAQ605,BMXBMI,LBXGLU,DIQ010,LBXGLT,LBXIN
0,Adult,61.0,2.0,2.0,35.7,110.0,2.0,150.0,14.91
1,Adult,26.0,2.0,2.0,20.3,89.0,2.0,80.0,3.85
2,Adult,16.0,1.0,2.0,23.2,89.0,2.0,68.0,6.14
3,Adult,32.0,1.0,2.0,28.9,104.0,2.0,84.0,16.15
4,Adult,38.0,2.0,1.0,35.9,103.0,2.0,81.0,10.92
5,Adult,50.0,1.0,2.0,23.6,110.0,2.0,100.0,6.08
6,Adult,14.0,1.0,2.0,38.7,94.0,2.0,202.0,21.11
7,Adult,57.0,2.0,2.0,38.3,107.0,2.0,164.0,20.93
8,Senior,75.0,1.0,2.0,38.9,89.0,2.0,113.0,17.47
9,Adult,43.0,1.0,1.0,28.9,90.0,2.0,95.0,3.24


In [7]:
# convert ages into ints
df["RIDAGEYR"] = df["RIDAGEYR"].astype(int)
df = df.rename(columns={"RIDAGEYR": "Age"})

In [8]:
# turn values of RIAGENDR into Male/Female
gender_map = {
    1.0: 0, # Male
    2.0: 1  # Female
}

df["RIAGENDR"] = df["RIAGENDR"].map(gender_map)
df = df.rename(columns={"RIAGENDR": "Gender"})
#df.head(3)

In [9]:
df["PAQ605"] = df["PAQ605"].map({
    1.0: 1, # moderate/vigorous physical activity
    2.0: 0 # no physical activity
})
df = df.rename(columns = {"PAQ605":"Physical Activity"})
df.head(3)

,age_group,Age,Gender,Physical Activity,BMXBMI,LBXGLU,DIQ010,LBXGLT,LBXIN
0,Adult,61,1,0.0,35.7,110.0,2.0,150.0,14.91
1,Adult,26,1,0.0,20.3,89.0,2.0,80.0,3.85
2,Adult,16,0,0.0,23.2,89.0,2.0,68.0,6.14


In [10]:
df["Physical Activity"] = df["Physical Activity"].fillna("Unknown")

In [11]:
df["Physical Activity"].value_counts(dropna=False)

Physical Activity
0.0        1868
1.0         409
Unknown       1
Name: count, dtype: int64

A single record presenting an undefined value (‘Unknown’) in the Physical Activity attribute was removed during the data cleaning phase, as the value does not represent a meaningful category and would trivially violate anonymity constraints without providing analytical value.

In [12]:
df = df[df["Physical Activity"] != "Unknown"]

In [13]:
df["age_group"] = df["age_group"].map({"Adult": 0, "Senior": 1})

In [14]:
df = df.rename(columns = {"BMXBMI":"BMI"})
df = df.rename(columns = {"LBXGLU":"Glucose Level"})
df = df.rename(columns={"LBXGLT": "Oral Glucose Test"})
df = df.rename(columns={"LBXIN": "Insulin Level"})

In [15]:
# from the documentation of the DIQ010 questionnaire
# ordinal variable
diq_map = {
    1.0: 2, # Diabetes
    3.0: 1, # Borderline
    2.0: 0  # NO
}

df["DIQ010"] = df["DIQ010"].map(diq_map)
df = df.rename(columns={"DIQ010": "Diabetes"})
#df.head(10)

In [16]:
df.isna().mean().sort_values(ascending=False)

age_group            0.0
Age                  0.0
Gender               0.0
Physical Activity    0.0
BMI                  0.0
Glucose Level        0.0
Diabetes             0.0
Oral Glucose Test    0.0
Insulin Level        0.0
dtype: float64

There's a low missing rate for Physical Activity, while the other attributes have no missing values at all (the dataset was already pre-processed elsewhere).
The missing rate in question is very low so it will not affect utility metrics. We should still explicitly encode it as missing (better for synthetic data generation)

In [17]:
df.describe()

,age_group,Age,Gender,BMI,Glucose Level,Diabetes,Oral Glucose Test,Insulin Level
count,2277.000000,2277.000000,2277.000000,2277.000000,2277.000000,2277.000000,2277.000000,2277.000000
mean,0.159859,41.791831,0.511638,27.958191,99.548529,0.043917,115.000439,11.838129
std,0.366556,20.159615,0.499974,7.249129,17.892423,0.245887,47.060374,9.719642
min,0.000000,12.000000,0.000000,14.500000,63.000000,0.000000,40.000000,0.140000
25%,0.000000,24.000000,0.000000,22.800000,91.000000,0.000000,87.000000,5.860000
50%,0.000000,41.000000,1.000000,26.800000,97.000000,0.000000,105.000000,9.040000
75%,0.000000,58.000000,1.000000,31.200000,104.000000,0.000000,130.000000,14.440000
max,1.000000,80.000000,1.000000,70.100000,405.000000,2.000000,604.000000,102.290000


Descriptive statistics indicate that all numeric attributes are complete and fall within clinically plausible ranges. Age spans from 12 to 80 years, with a balanced distribution. BMI and laboratory measurements exhibit extreme but realistic values. No implausible or out-of-range values were observed, and thus no outlier removal was applied.

In [18]:
for c in ["Gender", "Physical Activity", "Diabetes", "age_group"]:
    print(df[c].value_counts(normalize=True))
    print("\n")

Gender
1    0.511638
0    0.488362
Name: proportion, dtype: float64


Physical Activity
0.0    0.820378
1.0    0.179622
Name: proportion, dtype: float64


Diabetes
0    0.965305
1    0.025472
2    0.009223
Name: proportion, dtype: float64


age_group
0    0.840141
1    0.159859
Name: proportion, dtype: float64




Categorical attributes exhibit varying degrees of imbalance. Gender is nearly uniformly dstributed, while physical activity and diabetes status show skewness. These rare categories are known to contribute significantly to re-identification and attribute disclosure risk and were therefore explicitly considered in the privacy analysis.

The id attribute was excluded from the feature set, as it represents a technical identifier with no semantic relationship to the target variable and may introduce spurious correlations or information leakage.

In [19]:
df.to_csv("NHANES_1.csv", index=False)

## Data quality

Now that we have the polished dataset we can start by assessing whether it yields good performance when a classification task is performed: let's consider as target the Diabetes attribute: it's a sensitive attribute, clinically meaningful, strongly related to many other attributes and not trivially predictable. It's ideal for testing even when we'll consider the synthetic dataset and the traditionally-anonymized one.

### Target = Diabetes

#### 3-class classification

In [20]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("NHANES_1.csv")

TARGET = "Diabetes"
# since we're considering the age, we can avoid the age_group attribute, it's redundant
FEATURES = [c for c in df.columns if c not in [TARGET]]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))

Dummy baseline
Accuracy: 0.9649122807017544
Macro-F1: 0.3273809523809524


A DummyClassifier predicting the majority class achieves high accuracy due to strong class imbalance, but yields a very low Macro-F1 score, indicating no discriminative capability for minority diabetes classes. This baseline highlights the inadequacy of accuracy alone and motivates the use of balanced metrics for utility evaluation.

In [22]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import OneHotEncoder # to convert categorical values into binary indicator variables
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "age_group"]   #0/1
num_features = ["Age", "BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        class_weight = "balanced",
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.756578947368421
Macro-F1: 0.36002837256859205


When modeling diabetes status as a three-class problem, even a class-balanced logistic regression fails to substantially outperform a majority-class baseline in terms of Macro-F1. This indicates that the minority classes are difficult to separate given the available features, reflecting both severe class imbalance and intrinsic overlap in clinical measurements.

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.9649122807017544
Macro-F1: 0.3273809523809524


From the Macro-F1 metric we can see that the model is still dominated by the majority class. It has a slightly lower accuracy than the Dummy classifier, which means that it occasionally predicts minority classes, but not consistently or correctly enough. Basically, it means the 3-class diabetes prediction task has intrinsically low separability given the available features and extreme class imbalance.

#### Binary classification

We should now try to model the classification as a binary classification problem to reduce noise, reduce class overlap. <br>
To achieve it, we'll merge Yes + Borderline into the Attribute Diabetes:

In [25]:
import pandas as pd

df = pd.read_csv("NHANES_1.csv")
df["Diabetes_bin"] = df["Diabetes"].replace({
    0: 0, # NO
    1: 1, # Borderline
    2: 1 # Diabetes
})
df = df.drop(columns="Diabetes")

In [26]:
df["Diabetes_bin"].value_counts()

Diabetes_bin
0    2198
1      79
Name: count, dtype: int64

In [27]:
TARGET = "Diabetes_bin"
FEATURES = [c for c in df.columns if c not in [TARGET]]

X = df[FEATURES]
y = df[TARGET]

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [29]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))


Dummy baseline
Accuracy: 0.9649122807017544
Macro-F1: 0.49107142857142855


In [30]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "age_group"]   #0/1
num_features = ["Age", "BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Logistic Regression
Accuracy: 0.9605263157894737
Macro-F1: 0.4899328859060403


In [32]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Random Forest
Accuracy: 0.9649122807017544
Macro-F1: 0.49107142857142855


##### Results


|  | Accuracy | Macro-F1 |
| :--- | :--- | :--- | 
| **Dummy Baseline** | 0.9649 | 0.4911| 
| **Linear Regression** | 0.9605 | 0.4899|
| **Random Forests** | 0.9649 |0.4911 |

In the binary diabetes classification task, all evaluated models achieve nearly identical Accuracy and Macro-F1 scores, including the Dummy baseline, Logistic Regression, and Random Forest. This behavior reflects the strong class imbalance present in the dataset, where non-diabetic individuals account for approximately 96.5% of the records. As a consequence, a trivial majority-class predictor already attains very high accuracy and a Macro-F1 close to 0.49, as the minority class receives an F1-score of zero and is averaged with the majority class. <br>
More complex models fail to improve upon this baseline, indicating that the available features do not provide sufficient discriminative power to reliably identify diabetic cases. <br>
These results suggest that, in this setting, diabetes prediction is constrained more by data characteristics than by model capacity, and that limited but realistic predictive utility is present in the original dataset.

### Target = age_group

Although the dataset includes the continuous variable Age, it is labeled as an auxiliary (‘Other’) attribute in the UCI description. Since age_group is directly derived from Age, including this variable results in a near-deterministic prediction task. Therefore, experiments are conducted both with and without Age to distinguish between an upper-bound utility scenario and a realistic inference setting

#### with Age

##### 3-class classification


In [33]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("NHANES_1.csv")
#df.columns

In [34]:
TARGET = "age_group"
FEATURES = [c for c in df.columns if c not in [TARGET]]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))

Dummy baseline
Accuracy: 0.8399122807017544
Macro-F1: 0.4564958283671037


In [36]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import OneHotEncoder # to convert categorical values into binary indicator variables
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "Diabetes"] 
num_features = ["Age", "BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)

In [37]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        class_weight = "balanced",
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.9824561403508771
Macro-F1: 0.9687489291710927


In [38]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Accuracy: 1.0
Macro-F1: 1.0


##### Binary classification

Let's see how the classification changes when Diabetes is binned into the binary class

In [39]:
import pandas as pd

df = pd.read_csv("NHANES_1.csv")
df["Diabetes_bin"] = df["Diabetes"].replace({
    0: 0, # NO
    1: 1, # Borderline
    2: 1 # Diabetes
})
df = df.drop(columns="Diabetes")

In [40]:
TARGET = "age_group"
FEATURES = [c for c in df.columns if c not in [TARGET]]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [41]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))

Dummy baseline
Accuracy: 0.8399122807017544
Macro-F1: 0.4564958283671037


In [42]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import OneHotEncoder # to convert categorical values into binary indicator variables
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "Diabetes_bin"] 
num_features = ["Age", "BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        class_weight = "balanced",
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.9824561403508771
Macro-F1: 0.9687489291710927


In [44]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Accuracy: 1.0
Macro-F1: 1.0


#### without age

##### 3-class classification

In [45]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("NHANES_1.csv")
df = df.drop(columns = "Age")
df.columns
#df["age_group"].value_counts(normalize=True)

Index(['age_group', 'Gender', 'Physical Activity', 'BMI', 'Glucose Level',
       'Diabetes', 'Oral Glucose Test', 'Insulin Level'],
      dtype='object')

In [46]:
df.isna().sum()

age_group            0
Gender               0
Physical Activity    0
BMI                  0
Glucose Level        0
Diabetes             0
Oral Glucose Test    0
Insulin Level        0
dtype: int64

In [47]:
TARGET = "age_group"
FEATURES = [c for c in df.columns if c not in TARGET]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [48]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))

Dummy baseline
Accuracy: 0.8399122807017544
Macro-F1: 0.4564958283671037


In [49]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import OneHotEncoder # to convert categorical values into binary indicator variables
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "Diabetes"] 
num_features = ["BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)

In [50]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        class_weight = "balanced",
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.6885964912280702
Macro-F1: 0.5963394259781064


In [51]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Accuracy: 0.8333333333333334
Macro-F1: 0.565561851276137


##### Binary classification

In [52]:
import pandas as pd

df = pd.read_csv("NHANES_1.csv")
df["Diabetes_bin"] = df["Diabetes"].replace({
    0: 0, # NO
    1: 1, # Borderline
    2: 1 # Diabetes
})
df = df.drop(columns=["Diabetes", "Age"])
df.to_csv("NHANES_2.csv", index = False)

In [53]:
TARGET = "age_group"
FEATURES = [c for c in df.columns if c not in TARGET]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [54]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy baseline")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Macro-F1:", f1_score(y_test, y_pred_dummy, average="macro"))


Dummy baseline
Accuracy: 0.8399122807017544
Macro-F1: 0.4564958283671037


In [55]:
from sklearn.compose import ColumnTransformer # to apply different preprocessing steps to different columns
from sklearn.preprocessing import OneHotEncoder # to convert categorical values into binary indicator variables
from sklearn.preprocessing import StandardScaler

cat_features = ["Gender", "Physical Activity", "Diabetes_bin"] 
num_features = ["BMI", "Glucose Level", "Oral Glucose Test", "Insulin Level"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features), # avoid hot encoding binary variables
        ("num", StandardScaler(), num_features),
    ],
    remainder = "drop"
)

In [56]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        class_weight = "balanced",
        n_jobs=-1
    ))
])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))


Accuracy: 0.6907894736842105
Macro-F1: 0.6002660913787636


In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Accuracy: 0.8333333333333334
Macro-F1: 0.5575980392156863


#### **Results**

| Feature Context | Classification Task | Model | Accuracy | Macro-F1 |
| :--- | :--- | :--- | :--- | :--- |
|  |  | Dummy Baseline | 0.8399 | 0.4565 |
| **With Age** | 3-class classification | Logistic Regression | 0.9824 | 0.9687 |
| | | Random Forest | 1.0 | 1.0 |
| **With Age** | Binary classification| Logistic Regression | 0.9824 | 0.9687 |
| | | Random Forest | 1.0 | 1.0 |
| **Without Age** | 3-class classification | Logistic Regression | 0.6886 | 0.5963 |
| | | Random Forest | 0.8333 | 0.5655 |
| **Without Age** | Binary classification | Logistic Regression | 0.6908 | 0.6003 |
| | | Random Forest | 0.8333 | 0.5575 |

* **with Age** we see a near-deterministic prediction: age_group is a direct discretization of Age, the models are basically learning thresholds on a single variable, and Random Forests can memorize exact split boundaries, which leads to perfect scores. <br>
The fact that Logistic Regression already reaches ~99% shows that the relationship is nearly linear, so it means that the task is structurally easy, not that the model is extremely good
* **without Age** we can see a realistic and informative task: by removing the atttribute age, the performance drops and basically models rely on only BMI, glucose, insulin and physical activity. These are correlated but noisy proxis of age.
    * Dummy model: 
        * Accuracy ≈ 0.84 Macro-F1 ≈ 0.46 (Identical with and without Age, with Diabetes as 3-class an binary)
        * There is strong class imbalance in age_group, the models must beat Macro-F1, not just accuracy $\rightarrow$ in our case all models beat the baseline, so real learning is happening.
    * 3-class classification:
        * Logistic Regression:
            * has a lower accuracy than Random Forest, but reaches a higher Macro-F1
            * The model is linear and more balance across age groups, less biased toward majority classes and has better minority-class sensitivity.
        * Random Forest:
            * Has higher accuracy but lower Macro-F1
            * It likely over-predicts dominant age groups but it captures non-linear interactions and suffers from class imbalance effects.
        
    * Binary classification: is the same as the 3-classification results, but with slightly higher valeus for all metrics $\rightarrow$ we'll consider this setting for the synthetic data generation

When the continuous variable Age is included, classification performance approaches perfection, indicating a near-deterministic relationship between Age and the target age_group. Conversely, removing Age leads to a substantial performance decrease, reflecting a more realistic inference scenario in which age must be inferred indirectly from correlated clinical variables. This gap highlights the importance of excluding semantically upstream attributes to avoid trivial prediction and information leakage.
        

From what we've observed from the Data Quality section, we'll consider the binned version of the Diabetes attribute and we'll exclude the Age attribute to avoid implementing a trivial classification task for later.